In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT.parent / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from metrics_group_merger import MetricsGroup, PCMDIRun, merge_metrics_group


In [ ]:
# Shared merge setup.
CASE_ID = "v20260212"
RUN_TYPE = "model_vs_obs"

DIAG_ROOT = Path("/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le")
METRICS_DATA_ROOT = Path("/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data")

ENABLE_CLIM = True
ENABLE_MOVS = True
ENABLE_ENSO = True
STRICT = True
VERBOSE = True
DRY_RUN = False
CLEAN_OUTPUT = True

MIPS = ["e3sm"]
EXPS = ["historical"]

ALL_MOVS_MODES = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
ALL_MOVS_OBSES = (
    "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,"
    "HadISST,HadISST,HadISST"
)
ENSO_COLLECTIONS = "ENSO_perf,ENSO_tel,ENSO_proc"


In [ ]:
# Define the raw run groups to merge.
# Use model_pattern for conventional directory families, or member_names for exact arbitrary run directories.
TEST_RUNS = [
    PCMDIRun(
        case="20231209.v3.LR.piControl-spinup.chrysalis",
        model_name="EAMXX-ne256_coupled-test",
        metrics_case_id="v20260531",
        output_name="EAMXX-ne256-piControl",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
    PCMDIRun(
        case="20250906.wcycl1850.ne120pg2_r025_RRSwISC6to18E3r5.test6.1.chrysalis",
        model_name="v3-HR_test6-1",
        metrics_case_id="v20260601",
        output_name="v3-HR-test6-1",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
    PCMDIRun(
        case="20260204.ne256.WCYCLXX1850.SOI",
        model_name="EAMXX-ne256_coupled-test",
        metrics_case_id="v20260601",
        output_name="EAMXX-ne256-SOI",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
    PCMDIRun(
        case="ne256pg2_ne256pg2.F20TR-SCREAMv1.July-1.spanc800.2xauto.acc150.n0032.test2.1",
        model_name="EAMXX_test2_1",
        metrics_case_id="v20260531",
        output_name="EAMXX-test2-1",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
]

GROUPS = [
    MetricsGroup(
        name="climo",
        model_pattern="*",
        runs=TEST_RUNS,
        mips=MIPS,
        exps=EXPS,
        case_id=CASE_ID,
        model_rename=("v3-LR", "v3-LR-Historical", True),
        movs_years=(1850, 2014),
        movs_modes=ALL_MOVS_MODES,
        movs_obses=ALL_MOVS_OBSES,
        enso_collections=ENSO_COLLECTIONS,
        enso_obses="Tropflux,Tropflux,Tropflux",
    ),
    MetricsGroup(
        name="historical",
        model_pattern="*",
        runs=[
            PCMDIRun(
                case="v3.LR.historical_0051",
                model_name="e3sm.historical.v3-LR_0051",
                www=DIAG_ROOT / "hist",
            ),
        ],
        mips=MIPS,
        exps=EXPS,
        case_id=CASE_ID,
        model_rename=("v3-LR", "v3-LR-Historical", True),
        movs_years=(1980, 2014),
        movs_modes=ALL_MOVS_MODES,
        movs_obses=ALL_MOVS_OBSES,
        enso_collections=ENSO_COLLECTIONS,
        enso_obses="Tropflux,Tropflux,Tropflux",
    ),
    MetricsGroup(
        name="future",
        model_pattern="*",
        runs=[
            PCMDIRun(
                case="v3.LR.historical_0051",
                model_name="e3sm.historical.v3-LR_0051",
                www=DIAG_ROOT / "future",
            ),
        ],
        mips=MIPS,
        exps=EXPS,
        case_id=CASE_ID,
        model_rename=("v3-LR", "v3-LR-Future", True),
        movs_years=(2015, 2049),
        movs_modes=ALL_MOVS_MODES,
        movs_obses=ALL_MOVS_OBSES,
        enso_collections=ENSO_COLLECTIONS,
        enso_obses="FAKE-FUTURE2,FAKE-FUTURE2,FAKE-FUTURE2",
    ),
]

# Example for arbitrary run names:
# GROUPS.append(
#     MetricsGroup(
#         name="custom",
#         source_root=DIAG_ROOT / "custom_runs",
#         model_pattern="*",
#         member_names=[
#             "20231209.v3.LR.piControl-spinup.chrysalis",
#             "20250906.wcycl1850.ne120pg2_r025_RRSwISC6to18E3r5.test6.1.chrysalis",
#             "20260204.ne256.WCYCLXX1850.SOI",
#             "ne256pg2_ne256pg2.F20TR-SCREAMv1.July-1.spanc800.2xauto.acc150.n0032.test2.1",
#         ],
#         mips=MIPS,
#         exps=EXPS,
#         case_id=CASE_ID,
#         model_rename=("v3-LR", "Custom", True),
#         movs_years=(1850, 2014),
#         movs_modes=ALL_MOVS_MODES,
#         movs_obses=ALL_MOVS_OBSES,
#         enso_collections=ENSO_COLLECTIONS,
#         enso_obses="Tropflux,Tropflux,Tropflux",
#     )
# )


In [ ]:
# Merge all configured groups into METRICS_DATA_ROOT.
for group in GROUPS:
    print(f"Merging group: {group.name}")
    merge_metrics_group(
        group,
        metrics_root=METRICS_DATA_ROOT,
        run_type=RUN_TYPE,
        enable_clim=ENABLE_CLIM,
        enable_movs=ENABLE_MOVS,
        enable_enso=ENABLE_ENSO,
        strict=STRICT,
        verbose=VERBOSE,
        dry_run=DRY_RUN,
        clean=CLEAN_OUTPUT,
    )
